# Prompting

## Review

We built a chatbot that can hold a long conversation.

* We summarized older messages to keep the context small
* We saved the graph state to SQLite, so the conversation survives a restart

## Goals

Up to now, we have built every agent by hand from nodes and edges.

Now, we'll use [`create_agent`](https://docs.langchain.com/oss/python/langchain/agents), which builds that same loop for us.

The docs define an agent as "a model calling tools in a loop until a given task is complete."

With the loop taken care of, the prompt is the main lever we have over what the model returns.

We'll look at four ways to use it:

* A basic prompt, with and without a system prompt
* Few-shot examples
* Structured prompts
* Structured output

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

We'll use [LangSmith](https://docs.langchain.com/langsmith/home) for [tracing](https://docs.langchain.com/langsmith/observability-concepts).

We'll log to the project set by `LANGSMITH_PROJECT` in the repo-root `.env`, which is `ai-engineering`.

## Basic prompting

We create an agent from nothing but a model.

The string `"groq:openai/gpt-oss-20b"` names the provider and the model, so `create_agent` can initialize the chat model for us.

We invoke the agent with a list of messages, just like the graphs we built by hand.

`response['messages'][0]` is our question and `response['messages'][1]` is the model's reply.

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model="groq:openai/gpt-oss-20b")

question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

The Moon doesn’t have a capital—it's not a country or a political entity. It’s a natural satellite of Earth, so it has no governmental structure or capital city.


A [system prompt](https://docs.langchain.com/oss/python/langchain/agents#system-prompt) gives the agent a role.

The docs describe system prompts as the way to "shape how the agent approaches tasks".

Let's make the agent a science fiction writer and ask the same question again.

The system prompt is sent with every model call but is not stored in `messages`, so the reply is still at index `1`.

In [3]:
system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

**Selene Prime – The Capital of the Moon**

*Location:*  
Selene Prime is perched on the western flank of Mare Tranquillitatis, the “Sea of Tranquility,” a vast basaltic plain that was once the site of humanity’s first touchdown. The city sits within a shallow basin carved by ancient volcanic activity, providing a natural basin that is both a defensive moat and a reservoir for the city’s internal water‑recycling systems.

*Structure and Architecture:*  
The city is a layered, dome‑enclosed megastructure that rises 1.5 km above the lunar surface. It is composed of three concentric rings:

1. **The Core Ring** – A subterranean complex that houses the political heart of the lunar government. Thick regolith walls, reinforced with carbon‑nanotube composites, provide radiation shielding and thermal stability. The Core contains the Lunar Assembly Hall, the Supreme Lunar Council chambers, and the central data hub that monitors the entire moon.

2. **The Habitat Ring** – A series of interconnec

## Few-shot examples

The model can still answer in whatever style it likes.

Few-shot prompting puts a few example questions and answers directly in the prompt.

The model picks up the pattern from the examples.

Here, every capital is a short invented name that echoes the planet's name.

In [4]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

Scifi Writer: Lunaris.


## Structured prompts

We can also describe the exact layout we want back.

This is still plain text, though.

Nothing forces the model to follow the layout, and our code would have to parse the reply to get at each field.

In [5]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

Name: Selene City  
Location: South‑Pole ice cap, near the Shackleton Basin  
Vibe: Icy, austere, efficient  
Economy: Ice mining, cryogenic research, orbital logistics hub


## Structured output

[Structured output](https://docs.langchain.com/oss/python/langchain/structured-output) removes the parsing step.

"Instead of parsing natural language responses, you get structured data in the form of JSON objects, Pydantic models, or dataclasses that your application can use directly."

We pass a Pydantic model as `response_format`.

`create_agent` picks a strategy for us:

* `ProviderStrategy` uses the provider's native structured output, where it is supported
* `ToolStrategy` otherwise has the model fill in the schema through a tool call

"The structured response is returned in the `structured_response` key of the agent's final state."

In [6]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model='groq:openai/gpt-oss-20b',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

CapitalInfo(name='Luna Prime', location="Equatorial region of Earth's Moon", vibe='Silvery, serene, high-tech', economy='Resource extraction, research, tourism')

`structured_response` is a `CapitalInfo` instance, not a string.

We can read each field as an attribute.

In [7]:
response["structured_response"].name

'Luna Prime'

Because the fields are typed, the rest of our code can use them directly.

In [8]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")

Luna Prime is a city located at Equatorial region of Earth's Moon
